In [1]:
!pip install -U sentence-transformers

In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset

/Users/rayyanzaid/Desktop/School/CSCI-566-DeepLearning/CSCI-566-Course-Project-DeepPrep-AI/csci-566-project-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def parse_transcript(transcript_str):
    """Parse '[00:01 - 00:11] text' lines into list of (timestamp, text)."""
    if not transcript_str or not transcript_str.strip():
        return []
    segments = []
    for line in transcript_str.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("[") and "]" in line:
            idx = line.index("]")
            timestamp = line[1:idx].strip()
            text = line[idx + 1 :].strip()
            if text:
                segments.append((timestamp, text))
        else:
            segments.append(("", line))
    return segments


In [4]:
"""

Purpose: This function will go through the timestamp ranges in transcript_segments and compute the prosody features for the corresponding segments of the audio in the video.

Input: Video Path & Transcript Segments

    Example of transcript_segments:
    {
    "0:01 - 0:11": "Hello, my name is Rayyan.",
    "0:12 - 0:20": "Harish is working on this model with me.",
    }

Output: A 2D numpy array of shape (num_transcript_segments, num_prosody_features)
    Each row - corresponds to a transcript_segment (a timestamp range) 
    Each column - corresponds to a specific prosody feature (e.g., pitch, energy, speaking rate, etc.)

    Example of output array:
    [
        

    
    ]
"""


def parse_audio_return_prosody_features(video_path, transcript_segments)-> np.ndarray:
    # Placeholder
    print(f"Video Path: {video_path}")
    print(f"Transcript Segments: {transcript_segments}")
    return np.array([])

In [ ]:
print("hi")

In [5]:
# Load RecruitView dataset and parse transcripts by timestamp (one row per participant)

dataset = load_dataset("AI4A-lab/RecruitView")
train = dataset["train"]
    
# Use personality_score from dataset if present, else placeholder
personality_col = None
for col in ("personality_score", "overall_personality", "personality"):
    if col in train.column_names:
        personality_col = col
        break

# Build table: one row per participant; transcript_segments = { "0:01 - 0:11": "words", ... }
rows = []
for participant_id in range(len(train)):

    # --- START I/O Section ---
    # INPUTS from RecruitView Dataset
    transcript_str = train["transcript"][participant_id]
    video_file = train["video"][participant_id]
    video_path = video_file._hf_encoded['path']

    # OUTPUT 
    score = train[personality_col][participant_id] if personality_col else None

    # --- END I/O Section ---


    # --- START Parsing Section ---

    # Parsing Transcript 
    segments = parse_transcript(transcript_str)
    transcript_segments = {ts: text for ts, text in segments}

    # Parsing Audio to Get Prosody Features
    prosodyFeaturesArray = parse_audio_return_prosody_features(video_path, transcript_segments)
    
    # --- END Parsing Section ---

    
    rows.append({
        "participant_id": participant_id,
        "transcript_segments": transcript_segments,
        "personality_score": score,
        "prosody_features" : prosodyFeaturesArray
    })

table = pd.DataFrame(rows)
print(f"Loaded {len(train)} participants (one row each).")
print("Table columns:", list(table.columns))
print("Example transcript_segments (first participant):", table["transcript_segments"].iloc[0])
table.head()

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
ERROR:tornado.general:SEND Error: Host unreachable


KeyboardInterrupt: 

In [ ]:
# Embed segment transcripts: one 2D array per participant (n_segments, embed_dim)
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# One list of segment texts per participant (order preserved from dict)
segment_texts_per_participant = [list(row["transcript_segments"].values()) for _, row in table.iterrows()]
# Flatten to encode all segments in one batch
all_segment_texts = [t for segs in segment_texts_per_participant for t in segs]
embeddings_flat = model.encode(all_segment_texts, show_progress_bar=True)

# Split back into 2D arrays: one (n_segments, embed_dim) per participant
sizes = [len(segs) for segs in segment_texts_per_participant]
splits = np.cumsum(sizes)[:-1]
table["transcript_embeddings"] = np.split(embeddings_flat, splits)

print(f"Total segments: {len(all_segment_texts)}. Embedding dim: {embeddings_flat.shape[1]}.")
print("Per-participant shapes (n_segments, embed_dim):", [e.shape for e in table["transcript_embeddings"].iloc[:3]])
table.head()

In [ ]:
print(table.head())